# Análise exploratória dos dados de câncer no Brasil

Dados provenientes do INCA (Instituto Nacional de Câncer) foram carregados para análise. Fonte dos dados: [INCA - Registro Hospitalar de Câncer](https://irhc.inca.gov.br/RHCNet/visualizaTabNetExterno.action) e [Kaggle](https://www.kaggle.com/dfsets/rafatrindade/onco-360?select=raw_inca_registro_hospitalar.parquet).

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

In [0]:
df_spark = spark.read.parquet('/Volumes/workspace/default/cancer_data/raw_inca_registro_hospitalar.parquet')
display(df_spark)

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
import re

def clean_tnm(value):
    if value is None:
        return None
    value = str(value).strip()
    if re.fullmatch(r'[0-4][0-3][0-1]', value):
        return value
    return None

clean_tnm_udf = udf(clean_tnm, StringType())

df_spark = df_spark.withColumn('TNM', clean_tnm_udf(col('TNM')))

In [0]:
df_spark = df_spark.withColumn('PTNM', clean_tnm_udf(col('PTNM')))

In [0]:
df_spark.filter(col('TNM').isNotNull()).count()

In [0]:
df_spark.filter(col('PTNM').isNotNull()).count()

In [0]:
parquet_file_path = '/Volumes/workspace/default/cancer_data/raw_inca_registro_hospitalar.parquet' 
df = pd.read_parquet(parquet_file_path, engine='pyarrow')

In [0]:
df.head(15)

In [0]:
df.info()

In [0]:
# Deleta colunas irrelevantes ou com muita informação faltando
df = df.drop(columns=['VALOR_TOT', 'DTTRIAGE', 'MUUH', 'CNES', 'DTINITRT', 'LOCTUPRO', 'LATERALI', 'DTPRICON', 'ANTRI', 'ESTCONJ', 'PROCEDEN', 'CLITRAT', 'CLIATEN', 'LOCALNAS', 'ESTADIAG', 'ESTADIAM', 'TNM', 'ANOPRIDI'])

In [0]:
df = df.replace('nan', np.nan)

In [0]:
df = df.replace({col: {9: np.nan, '9': np.nan} for col in df.columns if col != 'IDADE'}) # 9 = sem info
df['BASDIAGSP'] = df['BASDIAGSP'].replace({4: np.nan, '4': np.nan}) # 4 = sem info

In [0]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(x=missing_values.index, y=missing_values.values, palette='viridis')
plt.xticks(rotation=90)
plt.title('Número de Valores Faltantes por Coluna')
plt.ylabel('Número de Valores Faltantes')
plt.xlabel('Colunas')
plt.tight_layout()
plt.show()

In [0]:
len(df)

In [0]:
df = df.dropna(subset=[col for col in df.columns if col != 'DATAOBITO'])

In [0]:
len(df)

In [0]:
df.isnull().sum()

In [0]:
df2 = df

In [0]:
df2_spark = spark.createDataFrame(df2)
df2_spark.write.mode('overwrite').saveAsTable("after_nan_table")

## Coerência dos dados

In [0]:
df = spark.table("after_nan_table").toPandas()

In [0]:
# Convertendo colunas de dfs para o formato datetime
date_cols = ['DTDIAGNO', 'DATAINITRT', 'DATAOBITO', 'DATAPRICON']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            format='%d/%m/%Y',
            errors='coerce'
        )

In [0]:
# Investigando as colunas de df
print("DTDIAGNO (Data de Diagnóstico):")
print(f"  Min: {df['DTDIAGNO'].min()}")
print(f"  Max: {df['DTDIAGNO'].max()}")
print(f"  NaN: {df['DTDIAGNO'].isna().sum()}")

print("\nDATAINITRT (Data de Início de Tratamento):")
print(f"  Min: {df['DATAINITRT'].min()}")
print(f"  Max: {df['DATAINITRT'].max()}")
print(f"  NaN: {df['DATAINITRT'].isna().sum()}")

print("\nDATAOBITO (Data de Óbito):")
print(f"  Min: {df['DATAOBITO'].min()}")
print(f"  Max: {df['DATAOBITO'].max()}")
print(f"  NaN: {df['DATAOBITO'].isna().sum()}")

print("\nDATAPRICON (Data Primeira Consulta):")
print(f"  Min: {df['DATAPRICON'].min()}")
print(f"  Max: {df['DATAPRICON'].max()}")
print(f"  NaN: {df['DATAPRICON'].isna().sum()}")

In [0]:
print("Datas de diagnóstico problemáticas (< 1900 ou > 2024):")
diagnostico_invalido = df[(df['DTDIAGNO'] < '1900-01-01') | (df['DTDIAGNO'] > '2024-12-31')]
print(f"   Total: {len(diagnostico_invalido)} casos ({len(diagnostico_invalido)/len(df)*100:.2f}%)")
df = df.drop(diagnostico_invalido.index)

print("Datas de tratamento problemáticas (< 1900 ou > 2024):")
tratamento_invalido = df[(df['DATAINITRT'] < '1900-01-01') | (df['DATAINITRT'] > '2024-12-31')] 
print(f"   Total: {len(tratamento_invalido)} casos ({len(tratamento_invalido)/len(df)*100:.2f}%)")
df = df.drop(tratamento_invalido.index) 

print("Datas de óbito problemáticas (< 1900 ou > 2024):")
obito_invalido = df[(df['DATAOBITO'] < '1900-01-01') | (df['DATAOBITO'] > '2024-12-31')]
print(f"   Total: {len(obito_invalido)} casos ({len(obito_invalido)/len(df)*100:.2f}%)")
df = df.drop(obito_invalido.index) 

print("\nTratamento ANTES do diagnóstico:")
tratamento_antes = df[(df['DATAINITRT'].notna()) & (df['DTDIAGNO'].notna()) & (df['DATAINITRT'] < df['DTDIAGNO'])]
print(f"   Total: {len(tratamento_antes)} casos ({len(tratamento_antes)/len(df)*100:.2f}%)")
df = df.drop(tratamento_antes.index) 

print("\nÓbito ANTES do diagnóstico:")
obito_antes_diag = df[(df['DATAOBITO'].notna()) & (df['DTDIAGNO'].notna()) & (df['DATAOBITO'] < df['DTDIAGNO'])]
print(f"   Total: {len(obito_antes_diag)} casos ({len(obito_antes_diag)/len(df)*100:.2f}%)")
df = df.drop(obito_antes_diag.index) 

print("\nÓbito ANTES do tratamento:")
obito_antes_trat = df[(df['DATAOBITO'].notna()) & (df['DATAINITRT'].notna()) & (df['DATAOBITO'] < df['DATAINITRT'])]
print(f"   Total: {len(obito_antes_trat)} casos ({len(obito_antes_trat)/len(df)*100:.2f}%)")
df = df.drop(obito_antes_trat.index) 

print("\nDatas no futuro (> 2024-12-31):")
print(f"   DTDIAGNO: {(df['DTDIAGNO'] > '2024-12-31').sum()}")
print(f"   DATAINITRT: {(df['DATAINITRT'] > '2024-12-31').sum()}")
print(f"   DATAOBITO: {(df['DATAOBITO'] > '2024-12-31').sum()}")

In [0]:
df = df.dropna(subset=['DTDIAGNO', 'DATAINITRT'])
df.isnull().sum()

In [0]:
# Limpeza e tratamento da coluna IDADE como inteiro
df['IDADE'] = pd.to_numeric(df['IDADE'], errors='coerce')
df = df[(df['IDADE'].notna()) & (df['IDADE'] >= 0) & (df['IDADE'] <= 100)]
df['IDADE'] = df['IDADE'].astype(int)

In [0]:
df = df2

In [0]:
df.head(100)

In [0]:
def clean_tnm(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    # T: 0-4, N: 0-3, M: 0-1
    if re.fullmatch(r'[0-4][0-3][0-1]', value):
        return value
    
    return np.nan


df['PTNM'] = df['PTNM'].apply(clean_tnm)

In [0]:
df.isnull().sum()

In [0]:
#TODO: precisa verificar inconsistencias em todas as outras features

## Criando novas features

- separar idade em faixas etárias?
- separar datas em anos (ou em periodos?), manter meses e dias?

In [0]:
df['STATUS_VITAL'] = np.where(
    (df['ESTDFIMT'] == '6') | ((df['DATAOBITO'].notna())),
    0,
    1
)

# se a pessoa estiver viva: data final do dataset (dez/2023) - data do diagnóstico
# se estiver morta: data de óbito - data do diagnóstico
data_final = pd.to_datetime('2023-12-31')
df['TEMPO_SEGUIMENTO'] = np.where(
    df['STATUS_VITAL'] == 1,
    (data_final - df['DTDIAGNO']).dt.days,
    (df['DATAOBITO'] - df['DTDIAGNO']).dt.days
)

df['TEMPO_ATE_TRATAMENTO'] = (df['DATAINITRT'] - df['DTDIAGNO']).dt.days

#df['TEMPO_ATE_PRIMEIRA_CONSULTA'] = (df['DATAPRICON'] - df['DTDIAGNO']).dt.days #FIXME: revisar essa feature, pra mim n ta fazendo sentido n

In [0]:
#FIXME: os valores desses tempos deram meio estranhos, tem valores negativos, tem que limpar as datas. ex:
df['TEMPO_ATE_TRATAMENTO'].describe()

In [0]:
#NOTE: NÃO CONSEGUI TESTAR, MEU KERNEL MORRE ANTES
# Escolher PTNM > TNM > NaN
def choose_tnm(ptnm, tnm):
    if ptnm.notna():
        return str(ptnm).strip()
    if tnm.notna():
        return str(tnm).strip()
    return np.nan

df['TNM_final'] = df.apply(
    lambda row: choose_tnm(row['PTNM'], row['TNM']),
    axis=1
)

# Decodificar T, N, M
df['T_cat'] = df['TNM_final'].str[0]
df['N_cat'] = df['TNM_final'].str[1]
df['M_cat'] = df['TNM_final'].str[2]
df.loc[df['TNM_final'].isna(), ['T_cat', 'N_cat', 'M_cat']] = np.nan

df = df.drop(columns=['PTNM', 'TNM', 'TNM_final'])

In [0]:
#TODO: excluir features de datas desnecessarias

## Encode

Fazer one-hot encoding das variáveis categóricas selecionadas, como tnm, loctupri, sexo, etc

Para um modelo de rsf, os códigos de CID-O não devem ser usados como valores numéricos brutos, pois isso introduz uma ordem artificial sem significado clínico; em vez disso:

- tratar a variável como categórica, preferencialmente usando a LOCTUPRI (CID-O, 4 dígitos) por ser mais informativa. Caso o número de observações seja limitado, usar LOCTUDET (CID-O, 3 dígitos)
- agrupar categorias raras (por exemplo, localizações com baixa frequência) em uma classe “Outros”
- aplicar one-hot encoding 

In [0]:
#TODO : coisas acima para a feature de CID-O

In [0]:
#TODO: fazer encoding de todas outras variaveis necessarias

## Gráficos

In [0]:
map_sexo = {
    '1': 'Masculino',
    '2': 'Feminino'
}

map_tabagismo = {
    '1': 'Nunca',
    '2': 'Ex-consumidor',
    '3': 'Sim',
    '4': 'Não avaliado',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_alcool = map_tabagismo.copy()

map_racacor = {
    '1': 'Branca',
    '2': 'Preta',
    '3': 'Amarela',
    '4': 'Parda',
    '5': 'Indígena',
    '9': 'Sem informação'
}

map_estdfimt = {
    '1': 'Remissão completa',
    '2': 'Remissão parcial',
    '3': 'Doença estável',
    '4': 'Progressão',
    '5': 'Suporte terapêutico',
    '6': 'Óbito',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_pritrath = {
    '1': 'Nenhum',
    '2': 'Cirurgia',
    '3': 'Radioterapia',
    '4': 'Quimioterapia',
    '5': 'Hormonioterapia',
    '6': 'Transplante de medula óssea',
    '7': 'Imunoterapia',
    '8': 'Outros',
    '9': 'Sem informação'
}

map_basmaimp = {
    '1': 'Clínica',
    '2': 'Pesquisa clínica',
    '3': 'Exame por imagem',
    '4': 'Marcadores tumorais',
    '5': 'Citologia',
    '6': 'Histologia da metástase',
    '7': 'Histologia do tumor primário',
    '9': 'Sem informação'
}

map_exdiag = {
    '1': 'Exame clínico / Patologia clínica',
    '2': 'Exames por imagem',
    '3': 'Endoscopia / Cirurgia exploradora',
    '4': 'Anatomia patológica',
    '5': 'Marcadores tumorais',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_orienc = {
    '1': 'SUS',
    '2': 'Não SUS',
    '3': 'Conta própria',
    '8': 'Não se aplica',
    '9': 'Sem informação'
}

map_tp_caso = {
    '1': 'Analítico',
    '2': 'Não analítico',
}

map_status_vital = {
    0: 'Óbito',
    1: 'Vivo'
}

In [0]:
df['DTDIAGNO_Year'] = pd.to_datetime(df['DTDIAGNO'], errors='coerce').dt.year
obito_counts = df[df['DATAOBITO'].isna()].groupby('DTDIAGNO_Year').size()
plt.figure(figsize=(10, 6))
sns.barplot(x=obito_counts.index, y=obito_counts.values, palette='viridis')
plt.title("Count of nan in DATAOBITO by Year of DTDIAGNO")
plt.xlabel("Year of DTDIAGNO")
plt.ylabel("Count of nan in DATAOBITO")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['SEXO'].map(map_sexo).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Distribuição por Sexo')
axes[0].set_ylabel('Pacientes')

df['RACACOR'].map(map_racacor).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Distribuição por Raça/Cor')

df['IDADE'].dropna().plot(kind='hist', bins=20, ax=axes[2])
axes[2].set_title('Distribuição de Idade')
axes[2].set_xlabel('Idade')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['TABAGISM'].map(map_tabagismo).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Histórico de Tabagismo')

df['ALCOOLIS'].map(map_alcool).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Histórico de Alcoolismo')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['BASMAIMP'].map(map_basmaimp).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Base Mais Importante do Diagnóstico')

df['EXDIAG'].astype(str).str.strip().map(map_exdiag).fillna('Sem informação') \
    .value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Exames para Diagnóstico')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df['PRITRATH'].map(map_pritrath).value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Primeiro Tratamento Recebido')

df['ESTDFIMT'].map(map_estdfimt).value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Estado da Doença ao Final do Tratamento')

plt.tight_layout()
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

df.loc[df['STATUS_VITAL'] == 0, 'IDADE'].dropna().plot(
    kind='hist', bins=20, alpha=0.7, ax=axes[0]
)
axes[0].set_title('Idade - Não Óbito')
axes[0].set_xlabel('Idade')

df.loc[df['STATUS_VITAL'] == 1, 'IDADE'].dropna().plot(
    kind='hist', bins=20, alpha=0.7, ax=axes[1]
)
axes[1].set_title('Idade - Óbito')
axes[1].set_xlabel('Idade')

plt.tight_layout()
plt.show()

In [0]:
tpcaso = (
    df['TPCASO']
    .astype(str)
    .str.strip()
    .map(map_tp_caso)
    .fillna('Sem informação')
)

cross = pd.crosstab(tpcaso, df['STATUS_VITAL'].map(map_status_vital))
cross_prop = cross.div(cross.sum(axis=1), axis=0)

cross_prop.plot(
    kind='bar',
    stacked=True,
    figsize=(8, 5)
)

plt.title('Proporção de Óbito por Tipo de Caso')
plt.ylabel('Proporção')
plt.xlabel('Tipo de Caso')
plt.legend(title='Status Vital')
plt.tight_layout()
plt.show()

In [0]:
#TODO: mais graficos e análises